# 2ª Submissão: Extração de Features + LSTM

Este notebook contém o pipeline completo para a 2ª submissão: carregar os dados (incluindo datasets externos), aplicar a extração de features TF-IDF e Handcrafted, treinar o modelo 
LSTM e, por fim, processar e guardar as previsões do conjunto de teste num CSV.

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from datasets import load_dataset


## 1. Carregar Datasets

In [ ]:
def get_prof_dataset(n_lines: int = 10000) -> pd.DataFrame:
    try:
        dataset_path = "../data/dataset-exemplos.csv"
        df = pd.read_csv(dataset_path, sep=";")
        df["id"] = df["ID"]
    except:
        dataset_path = "../data/dataset_treino.csv"
        df = pd.read_csv(dataset_path)
        df["id"] = df.index
        df["Label"] = df["source_name"]

    n_lines = min(n_lines, len(df))
    return df[["id", "Text", "Label"]].sample(n_lines, random_state=42).reset_index(drop=True)

def get_otb_dataset(n_lines: int = 10000) -> pd.DataFrame:
    try:
        dataset = load_dataset("MLNTeam-Unical/OpenTuringBench", name="in_domain")
        df_train = dataset["train"].to_pandas()
        df_test = dataset["test"].to_pandas()
        df = pd.concat([df_train, df_test], ignore_index=True)

        mapping_classes = {
            "meta-llama": "Meta",
            "qwen": "OpenAI",
            "google": "Google",
            "anthropic": "Anthropic",
        }
        df["id"] = df["url"]
        df["Text"] = df["content"]
        df["Label"] = df["model"].apply(lambda x: mapping_classes.get(x.split("/")[0].lower(), "Others"))
        df = df[df["Label"] != "Others"]
        n_lines = min(n_lines, len(df))
        return df[["id", "Text", "Label"]].sample(n_lines, random_state=42).reset_index(drop=True)
    except:
        return pd.DataFrame(columns=["id", "Text", "Label"])

def get_atdp_dataset(n_lines: int = 10000) -> pd.DataFrame:
    try:
        dataset = load_dataset("artem9k/ai-text-detection-pile")
        df = dataset["train"].to_pandas()
        df["Text"] = df["text"]
        df = df[df["source"] == "human"]
        df["Label"] = df["source"].apply(lambda x: "Human" if x == "human" else "")
        n_lines = min(n_lines, len(df))
        return df[["id", "Text", "Label"]].sample(n_lines, random_state=42).reset_index(drop=True)
    except:
        return pd.DataFrame(columns=["id", "Text", "Label"])

def get_ap_dataset(n_lines: int = 3000) -> pd.DataFrame:
    try:
        dataset = load_dataset("Anthropic/persuasion")
        df = dataset["train"].to_pandas()
        df["id"] = df["worker_id"]
        df["Text"] = df["argument"]
        df["Label"] = df["source"].apply(lambda x: "Anthropic" if x.startswith("Claude") else "Human")
        n_lines = min(n_lines, len(df))
        return df[["id", "Text", "Label"]].sample(n_lines, random_state=42).reset_index(drop=True)
    except:
        return pd.DataFrame(columns=["id", "Text", "Label"])

def get_datasets() -> pd.DataFrame:
    df_prof = get_prof_dataset()
    df_otb = get_otb_dataset()
    df_atdp = get_atdp_dataset()
    df_ap = get_ap_dataset()

    df = pd.concat([df_prof, df_otb, df_atdp, df_ap], ignore_index=True)
    return df

df_full = get_datasets()
print(df_full["Label"].value_counts())


## 2. Funções de Extração de Features

In [ ]:
def preprocess_text(text):
    ###acabar preprocessamento 


## 3. Preparação do Pipeline e Treino

In [ ]:
# Preparar divisão de Treino/Validação
df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df_full) * 0.8)

train_texts = df_full["Text"][:split_idx].fillna("").astype(str).tolist()
train_labels = df_full["Label"][:split_idx].tolist()

val_texts = df_full["Text"][split_idx:].fillna("").astype(str).tolist()
val_labels = df_full["Label"][split_idx:].tolist()

y_train, mapping = encode_labels(train_labels)
y_val, _ = encode_labels(val_labels)

# Iniciar TF-IDF
vectorizer = build_vectorizer()
train_clean1 = [preprocess_text(t) for t in train_texts]
val_clean1 = [preprocess_text(t) for t in val_texts]

X_train_tfidf = vectorizer.fit_transform(train_clean1).toarray()
X_val_tfidf = vectorizer.transform(val_clean1).toarray()

# Handcrafted Features
train_clean2 = [preprocess_text_clean(t) for t in train_texts]
val_clean2 = [preprocess_text_clean(t) for t in val_texts]

X_train_hc, fnames = build_handcrafted_matrix(train_texts, train_clean2)
X_val_hc, _ = build_handcrafted_matrix(val_texts, val_clean2)

X_train_hc_std, X_val_hc_std, hc_mean, hc_std = standardize_train_test(X_train_hc, X_val_hc)

# Concatenar as matrizes
X_train_full = np.hstack([X_train_tfidf, X_train_hc_std])
X_val_full = np.hstack([X_val_tfidf, X_val_hc_std])

print("Shape do treino:", X_train_full.shape)


## 4. LSTM Dataset e Classificador

In [ ]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        if hasattr(X, "toarray"):
            X = X.toarray()
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, embed_dim=128, hidden_dim=128, n_classes=6):
        super().__init__()
        self.embedding = nn.Linear(input_dim, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.unsqueeze(1)
        output, (h, c) = self.lstm(x)
        return self.fc(h[-1])

train_dataset = TextDataset(X_train_full, y_train)
val_dataset = TextDataset(X_val_full, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMClassifier(input_dim=X_train_full.shape[1], n_classes=len(mapping)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
        
    acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/total:.4f} | Acc: {acc:.4f}")


## Gravacao do Modelo e Variáveis

In [ ]:
import pickle
import os
os.makedirs('../models/subm2_model', exist_ok=True)

torch.save(model.state_dict(), '../models/subm2_model/lstm_model.pt')
with open('../models/subm2_model/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
with open('../models/subm2_model/hc_stats.pkl', 'wb') as f:
    pickle.dump({'mean': hc_mean, 'std': hc_std}, f)
with open('../models/subm2_model/mapping.pkl', 'wb') as f:
    pickle.dump(mapping, f)
print("Modelo e artefatos guardados com sucesso em ../models/subm2_model/")


## 5. Correr no Dataset de Teste e Guardar o CSV Final

In [ ]:
test_df = pd.read_csv('../data/dataset_teste.csv')

def format_submit(test_df, vectorizer, hc_mean, hc_std, mapping):
    test_texts = test_df['Text'].fillna("").astype(str).tolist()
    
    test_clean1 = [preprocess_text(t) for t in test_texts]
    X_test_tfidf = vectorizer.transform(test_clean1).toarray()
    
    test_clean2 = [preprocess_text_clean(t) for t in test_texts]
    X_test_hc, _ = build_handcrafted_matrix(test_texts, test_clean2)
    X_test_hc_std = (X_test_hc - hc_mean) / hc_std
    
    X_test_full = np.hstack([X_test_tfidf, X_test_hc_std])

    test_dataset = TextDataset(X_test_full, np.zeros(len(X_test_full)))
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    model.eval()
    all_preds = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
    
    submission_df = test_df.copy()
    submission_df['predicted_class'] = all_preds
    
    reverse_mapping = {v: k for k, v in mapping.items()}
    submission_df['predicted_source'] = submission_df['predicted_class'].map(reverse_mapping)
    
    return submission_df

submission_df = format_submit(test_df, vectorizer, hc_mean, hc_std, mapping)

csv_out = '../subm1/subm2-MIA-B.csv'
submission_df.to_csv(csv_out, index=False)
print(f"Predições criadas no ficheiro {csv_out}")
submission_df[['Text', 'predicted_source']].head()
